# SAE pilot: does a TopK SAE beat PCA-30 as the text transform for LightGBM?

Runtime: **GPU (T4 is enough)**. Total time ≈ 1.5–2.5 h. Every step caches its output to Drive, so after a disconnect just re-run all cells.

1. Setup (repo + dependencies)
2. Train the SAE on a generic text corpus (no benchmark data → no leakage)
3. Run LightGBM × 3 datasets × 6 folds × 5 text encoders
4. Summarize: paired deltas vs the official Joint Frozen baseline (`e5-small` = E5 → PCA-30)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/sae_pilot'   # checkpoints + results persist here
import os; os.makedirs(WORK, exist_ok=True)

## 1. Setup
Upload `multabench_sae.zip` to `MyDrive/sae_pilot/` first (it is the official MulTaBench repo at commit `cf189b7` plus the SAE changes).

In [ ]:
%cd /content
!rm -rf MulTaBench && unzip -q -o $WORK/multabench_sae.zip -d /content && ls MulTaBench
%cd /content/MulTaBench
!pip install -q peft skrub kagglehub python-dotenv "autogluon.core" lightgbm datasets
import sys; sys.path.insert(0, '/content/MulTaBench')
os.environ['MULTABENCH_SAE_DIR'] = f'{WORK}/sae_checkpoints'

Kaggle: the benchmark datasets are public. If the download asks for credentials, set them here.

In [ ]:
# os.environ['KAGGLE_USERNAME'] = '...'
# os.environ['KAGGLE_KEY'] = '...'
!python -m pytest tests/test_sae.py -q 2>&1 | tail -3

## 2. Train the SAE
~300k texts from Wikitext, AG News, Yelp and DBpedia, embedded with E5-small-v2 in the same `passage: <col>: <text>` format the benchmark uses.
Watch the held-out **FVU** (fraction of variance unexplained): trained should be well below the random control. Rough target: < 0.3.
Also writes the random-SAE control checkpoint.

In [ ]:
!python train_sae.py --n_texts 300000 --n_latents 4096 --k 16 --epochs 20 --out_dir $MULTABENCH_SAE_DIR
!cat $MULTABENCH_SAE_DIR/e5-small-topk_stats.json

## 3. Run the pilot
Three datasets, one per task type. Results go to Drive one JSON per run; finished runs are skipped on re-run.

In [ ]:
!python sae_pilot.py run --model light \
    --datasets BIN_TEXT_FAKE_JOB_POSTING MUL_TEXT_WINE_REVIEW REG_TEXT_BOOK_READABILITY \
    --out_dir $WORK/runs_sae_pilot 2>&1 | grep -E "fold [0-9] |Error|error" 

## 4. Summary

In [ ]:
!python sae_pilot.py summarize --out_dir $WORK/runs_sae_pilot

### How to read it
- **e5-small-sae(-bin) vs e5-small**: the actual question.
- **vs e5-small-full**: if full-384 does as well as the SAE, the gain is just "more columns", not sparsity.
- **vs e5-small-rsae-bin**: if a random sparse code does as well, the gain is not from what the SAE learned.

Go/no-go: the SAE should beat the baseline on most datasets **and** beat both controls. If not, pivot early (Matryoshka / supervised term / oblique splits).